# 🛒 Product Return Prediction System
**Author:** Arushi Garg             
**Tools:** Python, Pandas, Scikit-learn, XGBoost, Matplotlib

## 🎯 Objective
Predict whether an e-commerce order will be returned using customer,
product, and order data — helping businesses reduce return costs.

## 📌 Dataset
15,000 Indian e-commerce orders with 19 features including product
category, discount %, delivery days, payment method, and more.

Install Libraries

In [ ]:
!pip install xgboost -q


Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve,
                              f1_score, precision_score, recall_score)
from xgboost import XGBClassifier
import pickle

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
COLORS = ['#2ECC71','#E74C3C','#3498DB','#F39C12','#9B59B6','#1ABC9C','#E67E22']

print("✅ All libraries imported successfully!")


Upload Dataset

In [ ]:
# Upload the CSV file in Colab, then:
from google.colab import files
uploaded = files.upload()  # upload ecommerce_product_returns.csv

import pandas as pd
df = pd.read_csv('ecommerce_product_returns.csv')
print(df.shape)
df.head()

Load & Preview Data

In [ ]:
df = pd.read_csv('ecommerce_product_returns.csv')

print('='*55)
print('  ✅  Dataset Loaded Successfully!')
print('='*55)
print(f'  Rows        : {df.shape[0]:,}')
print(f'  Columns     : {df.shape[1]}')
print(f'  Returns     : {df["is_returned"].sum():,}  ({df["is_returned"].mean()*100:.1f}%)')
print(f'  Not Returned: {(df["is_returned"]==0).sum():,}  ({(df["is_returned"]==0).mean()*100:.1f}%)')
print('='*55)
print()
print(df.head(10).to_string())

Data Info & Missing Values

In [ ]:
print("📋 Data Types & Missing Values")
print("-"*40)
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Basic Statistics:")
print(df.describe().round(2).to_string())

EDA CHART 1: Return Rate Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Return Rate Overview', fontsize=16, fontweight='bold')

# Pie chart
return_counts = df['is_returned'].value_counts()
axes[0].pie(return_counts, labels=['Not Returned','Returned'],
            colors=['#2ECC71','#E74C3C'], autopct='%1.1f%%',
            startangle=90, explode=(0,0.05), shadow=True)
axes[0].set_title('Overall Return Distribution', fontweight='bold')

# Category bar chart
cat_return = (df.groupby('product_category')['is_returned']
                .mean().sort_values(ascending=False) * 100)
bars = axes[1].bar(cat_return.index, cat_return.values, color=COLORS)
axes[1].set_title('Return Rate by Product Category', fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Return Rate (%)')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, cat_return.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.4, f'{val:.1f}%',
                 ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart1_return_overview.png', bbox_inches='tight')
plt.show()
print("💡 Insight: Fashion has the HIGHEST return rate (size/fit issues)!")

EDA CHART 2: Discount & Payment Method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['discount_bin'] = pd.cut(df['discount_percent'],
                             bins=[0,10,20,30,40,50,100],
                             labels=['0-10%','11-20%','21-30%','31-40%','41-50%','50%+'])
disc_return = df.groupby('discount_bin')['is_returned'].mean() * 100
c_grad = ['#27AE60','#F1C40F','#E67E22','#E74C3C','#C0392B','#8E1A0E']
bars = axes[0].bar(disc_return.index, disc_return.values,
                   color=c_grad, edgecolor='white', linewidth=1.5)
axes[0].set_title('Return Rate by Discount Level', fontweight='bold')
axes[0].set_xlabel('Discount Range')
axes[0].set_ylabel('Return Rate (%)')
for bar, val in zip(bars, disc_return.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3, f'{val:.1f}%',
                 ha='center', fontsize=9, fontweight='bold')

pay_return = (df.groupby('payment_method')['is_returned']
                .mean().sort_values(ascending=False) * 100)
bars2 = axes[1].barh(pay_return.index, pay_return.values, color='#3498DB')
axes[1].set_title('Return Rate by Payment Method', fontweight='bold')
axes[1].set_xlabel('Return Rate (%)')
for bar, val in zip(bars2, pay_return.values):
    axes[1].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart2_discount_payment.png', bbox_inches='tight')
plt.show()
print("💡 Insight: Higher discount = more returns! COD has the highest return rate.")

EDA CHART 3: Delivery Days & Customer Rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delivery_return = df.groupby('delivery_days')['is_returned'].mean() * 100
axes[0].plot(delivery_return.index, delivery_return.values,
             marker='o', color='#E74C3C', linewidth=2.5, markersize=7)
axes[0].fill_between(delivery_return.index, delivery_return.values,
                     alpha=0.2, color='#E74C3C')
axes[0].axvline(x=7, color='gray', linestyle='--', alpha=0.7, label='7-day mark')
axes[0].set_title('Return Rate by Delivery Days', fontweight='bold')
axes[0].set_xlabel('Delivery Days')
axes[0].set_ylabel('Return Rate (%)')
axes[0].legend()

rating_return = df.groupby('customer_rating')['is_returned'].mean() * 100
bars3 = axes[1].bar(rating_return.index, rating_return.values,
                    color=['#E74C3C','#E67E22','#F1C40F','#2ECC71','#27AE60'])
axes[1].set_title('Return Rate by Customer Rating', fontweight='bold')
axes[1].set_xlabel('Customer Rating (1=Low, 5=High)')
axes[1].set_ylabel('Return Rate (%)')
for bar, val in zip(bars3, rating_return.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3, f'{val:.1f}%',
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('chart3_delivery_rating.png', bbox_inches='tight')
plt.show()
print("💡 Insight: Late delivery (>7 days) and low ratings both increase returns!")

 EDA CHART 4: Customer Type, Warranty & Cities

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cust_return = (df.groupby('customer_type')['is_returned']
                 .mean().sort_values(ascending=False) * 100)
axes[0].bar(cust_return.index, cust_return.values,
            color=['#E74C3C','#F39C12','#2ECC71'])
axes[0].set_title('Return Rate by Customer Type', fontweight='bold')
axes[0].set_ylabel('Return Rate (%)')
for i, val in enumerate(cust_return.values):
    axes[0].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

warr_return = df.groupby('warranty_available')['is_returned'].mean() * 100
axes[1].bar(warr_return.index, warr_return.values,
            color=['#E74C3C','#2ECC71'], width=0.4)
axes[1].set_title('Return Rate: Warranty vs No Warranty', fontweight='bold')
axes[1].set_ylabel('Return Rate (%)')
for i, val in enumerate(warr_return.values):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

city_return = (df.groupby('customer_city')['is_returned']
                 .mean().sort_values(ascending=False).head(10) * 100)
axes[2].barh(city_return.index[::-1], city_return.values[::-1], color='#9B59B6')
axes[2].set_title('Top 10 Cities by Return Rate', fontweight='bold')
axes[2].set_xlabel('Return Rate (%)')

plt.tight_layout()
plt.savefig('chart4_customer_warranty_city.png', bbox_inches='tight')
plt.show()
print("💡 Insight: New customers return more. No-warranty products have higher return rates.")

EDA CHART 5: Correlation Heatmap

In [ ]:
numeric_cols = ['customer_age','product_price','discount_percent',
                'order_quantity','delivery_days','seller_rating',
                'customer_rating','is_returned']

corr = df[numeric_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('chart5_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("💡 Insight: discount_percent and delivery_days have the strongest correlation with returns!")

 EDA CHART 6: Return Reasons

In [ ]:
returned_df = df[df['is_returned'] == 1]
reason_counts = returned_df['return_reason'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].pie(reason_counts.values, labels=reason_counts.index,
            autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(reason_counts)),
            startangle=140)
axes[0].set_title('Distribution of Return Reasons', fontweight='bold')

top_cats = ['Fashion','Electronics','Home & Kitchen']
cat_reason_data = []
for cat in top_cats:
    top_reasons = (df[(df['product_category']==cat) & (df['is_returned']==1)]
                   ['return_reason'].value_counts().head(3))
    for reason, count in top_reasons.items():
        cat_reason_data.append({'Category': cat, 'Reason': reason, 'Count': count})

cr_df = pd.DataFrame(cat_reason_data)
cr_pivot = cr_df.pivot(index='Reason', columns='Category', values='Count').fillna(0)
cr_pivot.plot(kind='bar', ax=axes[1],
              color=['#3498DB','#E74C3C','#2ECC71'], edgecolor='white')
axes[1].set_title('Top Return Reasons by Category', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Category')

plt.tight_layout()
plt.savefig('chart6_return_reasons.png', bbox_inches='tight')
plt.show()
print("💡 Insight: Wrong Size/Fit is #1 in Fashion. Defective Product dominates Electronics returns.")

Feature Engineering

In [ ]:
df_model = df.copy()

# New features
df_model['actual_price']        = df_model['product_price'] * (1 - df_model['discount_percent']/100)
df_model['high_discount']       = (df_model['discount_percent'] >= 40).astype(int)
df_model['late_delivery']       = (df_model['delivery_days'] > 5).astype(int)
df_model['low_seller_rating']   = (df_model['seller_rating'] < 3.5).astype(int)
df_model['low_customer_rating'] = (df_model['customer_rating'] <= 2).astype(int)
df_model['premium_product']     = (df_model['product_price'] > 10000).astype(int)
df_model['order_month']         = pd.to_datetime(df_model['order_date']).dt.month
df_model['festival_season']     = df_model['order_month'].isin([10,11,12]).astype(int)
df_model['is_cod']              = (df_model['payment_method']=='Cash on Delivery').astype(int)

# Encode categorical columns
le = LabelEncoder()
cat_cols = ['customer_type','product_category','product_subcategory',
            'payment_method','shipping_type','warranty_available',
            'product_size_available','customer_city']
for col in cat_cols:
    df_model[col+'_enc'] = le.fit_transform(df_model[col])

print("✅ Feature Engineering Done!")
print(f"   Original features : {df.shape[1]}")
print(f"   New features added: 9")
print(f"   Total columns now : {df_model.shape[1]}")


Prepare X, y and Train-Test Split

In [ ]:
feature_cols = [
    'customer_age','product_price','discount_percent','order_quantity',
    'delivery_days','seller_rating','customer_rating',
    'actual_price','high_discount','late_delivery',
    'low_seller_rating','low_customer_rating','premium_product',
    'festival_season','is_cod','order_month',
    'customer_type_enc','product_category_enc','product_subcategory_enc',
    'payment_method_enc','shipping_type_enc','warranty_available_enc',
    'product_size_available_enc','customer_city_enc'
]

X = df_model[feature_cols]
y = df_model['is_returned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Train-Test Split Done!")
print(f"   Total features : {X.shape[1]}")
print(f"   Training set   : {X_train.shape[0]:,} rows")
print(f"   Test set       : {X_test.shape[0]:,} rows")
print(f"   Return rate    : {y.mean()*100:.1f}%")

 Train 5 Models & Compare

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost'             : XGBClassifier(n_estimators=100, random_state=42,
                                          eval_metric='logloss', verbosity=0)
}

results = []
trained_models = {}

print("Training models — please wait...\n")
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>8} {'ROC-AUC':>10} {'Precision':>11} {'Recall':>8}")
print("-"*76)

for name, model in models.items():
    use_scaled = (name == 'Logistic Regression')
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled  if use_scaled else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)

    results.append({'Model':name,'Accuracy':acc,'F1 Score':f1,
                    'ROC-AUC':auc,'Precision':prec,'Recall':rec})
    trained_models[name] = model

    print(f"{name:<25} {acc*100:>9.2f}% {f1*100:>7.2f}% {auc*100:>9.2f}% "
          f"{prec*100:>10.2f}% {rec*100:>7.2f}%")

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"\n🏆 Best Model: {best_model_name}  (ROC-AUC = {results_df.iloc[0]['ROC-AUC']*100:.2f}%)")


Model Comparison Bar Chart

In [ ]:
metrics     = ['Accuracy','F1 Score','ROC-AUC','Precision','Recall']
bar_colors  = ['#E74C3C','#2ECC71','#3498DB','#F39C12','#9B59B6']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, metric in enumerate(metrics):
    vals  = results_df[metric].values * 100
    names = results_df['Model'].values
    bars  = axes[i].bar(range(len(names)), vals, color=bar_colors, edgecolor='white')
    axes[i].set_title(metric, fontweight='bold', fontsize=11)
    axes[i].set_xticks(range(len(names)))
    axes[i].set_xticklabels([m.replace(' ','\n') for m in names], fontsize=7)
    axes[i].set_ylim([max(0, min(vals)-5), 100])
    axes[i].set_ylabel('%')
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.3, f'{val:.1f}',
                     ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('chart7_model_comparison.png', bbox_inches='tight')
plt.show()


 ROC Curves

In [ ]:
roc_colors = ['#E74C3C','#2ECC71','#3498DB','#F39C12','#9B59B6']
plt.figure(figsize=(8, 7))

for i, (name, model) in enumerate(trained_models.items()):
    use_scaled = (name == 'Logistic Regression')
    Xte   = X_test_scaled if use_scaled else X_test
    y_prob = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2.2, color=roc_colors[i],
             label=f'{name}  (AUC = {auc:.3f})')

plt.plot([0,1],[0,1],'k--', lw=1.5, alpha=0.6, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('chart8_roc_curves.png', bbox_inches='tight')
plt.show()


Confusion Matrix of Best Model

In [ ]:
use_scaled  = (best_model_name == 'Logistic Regression')
Xte         = X_test_scaled if use_scaled else X_test
y_pred_best = best_model.predict(Xte)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Not Returned','Predicted: Returned'],
            yticklabels=['Actual: Not Returned','Actual: Returned'],
            linewidths=1, linecolor='white', annot_kws={'size':14})
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart9_confusion_matrix.png', bbox_inches='tight')
plt.show()

print(f"\n📋 Classification Report — {best_model_name}:")
print(classification_report(y_test, y_pred_best,
      target_names=['Not Returned','Returned']))

Feature Importance

In [ ]:
rf_model    = trained_models['Random Forest']
importances = rf_model.feature_importances_
feat_df     = (pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
                 .sort_values('Importance', ascending=False).head(15))

plt.figure(figsize=(10, 7))
colors_imp = plt.cm.RdYlGn(np.linspace(0.8, 0.2, len(feat_df)))
plt.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1],
         color=colors_imp, edgecolor='white')
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Features — Random Forest Feature Importance',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart10_feature_importance.png', bbox_inches='tight')
plt.show()

print("🔑 Top 5 Most Important Features:")
for _, row in feat_df.head(5).iterrows():
    print(f"   {row['Feature']:<30}  →  {row['Importance']*100:.2f}%")


 Save Model

In [ ]:
with open('return_prediction_model.pkl','wb') as f:
    pickle.dump(best_model, f)
with open('scaler.pkl','wb') as f:
    pickle.dump(scaler, f)

print(f"✅ Model saved  →  return_prediction_model.pkl")
print(f"✅ Scaler saved →  scaler.pkl")


 Prediction Function (Test Any Order)

In [ ]:
def predict_return(order_details: dict) -> dict:
    """
    Predicts whether a product will be returned.
    Pass any order details as a dictionary.
    """
    customer_type_map = {'Loyal': 0, 'New': 1, 'Returning': 2}
    category_map      = {'Beauty & Health':0,'Books & Media':1,'Electronics':2,
                         'Fashion':3,'Home & Kitchen':4,'Sports & Fitness':5,'Toys & Baby':6}
    payment_map       = {'Cash on Delivery':0,'Credit Card':1,'Debit Card':2,
                         'EMI':3,'Net Banking':4,'UPI':5}
    shipping_map      = {'Express':0,'Free Shipping':1,'Same Day':2,'Standard':3}
    warranty_map      = {'No':0,'Yes':1}

    price    = order_details['product_price']
    discount = order_details['discount_percent']
    delivery = order_details['delivery_days']
    rating_c = order_details['customer_rating']
    rating_s = order_details['seller_rating']
    payment  = order_details['payment_method']
    category = order_details['product_category']
    month    = order_details.get('order_month', 6)

    features = [
        order_details.get('customer_age', 30),
        price, discount,
        order_details.get('order_quantity', 1),
        delivery, rating_s, rating_c,
        price * (1 - discount/100),       # actual_price
        int(discount >= 40),              # high_discount
        int(delivery > 5),                # late_delivery
        int(rating_s < 3.5),              # low_seller_rating
        int(rating_c <= 2),               # low_customer_rating
        int(price > 10000),               # premium_product
        int(month in [10,11,12]),         # festival_season
        int(payment=='Cash on Delivery'), # is_cod
        month,
        customer_type_map.get(order_details.get('customer_type','New'), 1),
        category_map.get(category, 2),
        0,                                # subcategory (simplified)
        payment_map.get(payment, 5),
        shipping_map.get(order_details.get('shipping_type','Standard'), 3),
        warranty_map.get(order_details.get('warranty_available','No'), 0),
        1,                                # size_available (simplified)
        0,                                # city (simplified)
    ]

    X_input = np.array(features).reshape(1, -1)
    prob    = best_model.predict_proba(X_input)[0][1]
    pred    = best_model.predict(X_input)[0]
    risk    = ('🔴 HIGH RISK'   if prob > 0.60 else
               '🟡 MEDIUM RISK' if prob > 0.35 else
               '🟢 LOW RISK')
    return {
        'will_return'        : bool(pred),
        'return_probability' : f'{prob*100:.1f}%',
        'risk_level'         : risk
    }


# ── Test 1: High Risk Order ──────────────────────────────────
order_high_risk = {
    'customer_age': 22, 'product_price': 2500, 'discount_percent': 60,
    'order_quantity': 1, 'delivery_days': 12, 'seller_rating': 2.8,
    'customer_rating': 2, 'payment_method': 'Cash on Delivery',
    'product_category': 'Fashion', 'customer_type': 'New',
    'shipping_type': 'Standard', 'warranty_available': 'No',
    'order_month': 11
}
r1 = predict_return(order_high_risk)
print("="*50)
print("🧪 TEST 1 — New customer, 60% discount, Fashion, COD, 12-day delivery")
print(f"   Will Return        : {r1['will_return']}")
print(f"   Return Probability : {r1['return_probability']}")
print(f"   Risk Level         : {r1['risk_level']}")

# ── Test 2: Low Risk Order ───────────────────────────────────
order_low_risk = {
    'customer_age': 35, 'product_price': 999, 'discount_percent': 5,
    'order_quantity': 1, 'delivery_days': 2, 'seller_rating': 4.8,
    'customer_rating': 5, 'payment_method': 'UPI',
    'product_category': 'Books & Media', 'customer_type': 'Loyal',
    'shipping_type': 'Express', 'warranty_available': 'Yes',
    'order_month': 6
}
r2 = predict_return(order_low_risk)
print()
print("🧪 TEST 2 — Loyal customer, 5% discount, Books, UPI, 2-day delivery")
print(f"   Will Return        : {r2['will_return']}")
print(f"   Return Probability : {r2['return_probability']}")
print(f"   Risk Level         : {r2['risk_level']}")
print("="*50)

 Project Summary

In [ ]:
best_row = results_df.iloc[0]
print()
print("="*60)
print("  🏁  PROJECT SUMMARY: PRODUCT RETURN PREDICTION SYSTEM")
print("="*60)
print(f"  Dataset         : 15,000 Indian e-commerce orders")
print(f"  Features Used   : {len(feature_cols)}")
print(f"  Models Tested   : {len(models)}")
print(f"  Best Model      : {best_model_name}")
print(f"  Accuracy        : {best_row['Accuracy']*100:.2f}%")
print(f"  ROC-AUC         : {best_row['ROC-AUC']*100:.2f}%")
print(f"  F1 Score        : {best_row['F1 Score']*100:.2f}%")
print("="*60)
print()
print("  📌 KEY BUSINESS INSIGHTS:")
print("  1. Fashion has the highest return rate (~42%)")
print("  2. Discounts above 40% significantly raise return risk")
print("  3. Cash on Delivery orders return more than digital payments")
print("  4. Delivery beyond 5 days increases return probability")
print("  5. New customers return ~6% more than loyal customers")
print("  6. Festival season (Oct–Dec) shows elevated return rates")
print("="*60)
print()
print("  ✅ Output Files:")
print("  - return_prediction_model.pkl")
print("  - scaler.pkl")
print("  - chart1 to chart10 (all saved as PNG)")
print("="*60)

## 📝 Business Recommendations

| Finding | Recommendation |
|---------|---------------|
| Fashion returns = 42% | Add size guide & virtual try-on |
| COD returns are highest | Offer COD only on orders < ₹500 |
| Discount >40% = high return | Cap max discount at 35% |
| Late delivery = more returns | Improve logistics for Tier-2 cities |
| New customers return more | Add return policy awareness at checkout |